# 🍽️ Gastronomia Computazionale - Analisi Completa delle Reti Culinarie

**Studentessa:** Aurora Felisari  
**Matricola:** 397867  
**Corso:** Laboratorio di Intelligenza Artificiale  
**Anno Accademico:** 2025/2026

---

## 📋 Panoramica del Progetto

Questo notebook presenta un'analisi multilivello delle strutture culinarie attraverso la teoria delle reti complesse:

### 1. Analisi Topologica delle Ricette

**Obiettivo:** Confronto quantitativo tra cucina tradizionale e d'avanguardia attraverso:

- **Misure di Centralità**: Identificazione degli ingredienti chiave
  - *Degree Centrality*: Connettività diretta
  - *Betweenness Centrality*: Ingredienti "ponte" tra comunità
  - *Closeness Centrality*: Vicinanza media agli altri nodi
  - *Eigenvector Centrality*: Influenza basata sulla qualità delle connessioni

- **Rilevamento delle Comunità**: Algoritmo di Louvain
  - Identifica gruppi di ingredienti fortemente interconnessi
  - Misura la modularità (compartimentazione) della rete

- **Proprietà Small-World**: Coefficienti di clustering e lunghezza dei cammini
  - Il coefficiente σ (sigma) quantifica il carattere "piccolo mondo"
  - σ > 1 indica proprietà small-world (alto clustering + cammini brevi)

**Ipotesi di Ricerca:** La cucina d'avanguardia presenta una topologia "Small World" più marcata rispetto alla cucina tradizionale, con maggiore interconnessione tra ingredienti di categorie diverse.

### 2. Analisi Chimica (Flavor Network)

**Fondamento Scientifico:** Utilizzo del dataset di Ahn et al. (2011) sui composti aromatici condivisi.

**Metodologia:**
- **Grafo Bipartito**: Relazioni Ingrediente ↔ Composto
  - 1.531 ingredienti
  - Centinaia di composti chimici volatili
  - Proiezione su rete ingrediente-ingrediente pesata

- **Ipotesi del Food Pairing**: Ingredienti che condividono composti aromatici si abbinano bene
  - Validazione attraverso similarità di Jaccard
  - Identificazione della "spina dorsale chimica" dei sapori

- **Analisi del Profilo Molecolare**:
  - Composti condivisi tra coppie di ingredienti
  - Pattern di abbinamento per categoria

### 3. Sistema Interattivo di Scoperta dei Bridge

**Innovazione Metodologica:** Analisi comparativa della connettività tra ingredienti

- **Identificazione Link Diretti**: Verifica connessioni immediate nel flavor network

- **Scoperta di Nodi Ponte**: Due approcci complementari
  - **Betweenness Centrality**: Identifica ingredienti "gatekeeper" che controllano i flussi
  - **Optimal Path Centrality**: Trova connettori basati sull'efficienza dei percorsi

- **Valutazione Qualitativa**: Confronto tra le due metriche per identificare i bridge più efficaci

---

## 🔬 Base Teorica

### Teoria delle Reti Complesse

Le **reti complesse** sono sistemi composti da nodi (ingredienti) e archi (relazioni) che mostrano proprietà emergenti non riducibili ai singoli componenti. Nel contesto culinario:

1. **Reti Small-World** (Watts & Strogatz, 1998):
   - Alto coefficiente di clustering locale
   - Breve distanza media tra nodi
   - Permette diffusione rapida di "informazione" (sapori)

2. **Modularità** (Newman & Girvan, 2004):
   - Misura la divisione in comunità
   - Valori alti (> 0.3) indicano struttura modulare forte
   - Nella gastronomia: gruppi di ingredienti che "si parlano"

3. **Centralità** (Freeman, 1978):
   - Misure di importanza dei nodi nella rete
   - Diverse prospettive rivelano ruoli diversi

### Food Pairing Hypothesis

L'**ipotesi del food pairing** (Ahn et al., 2011) sostiene che:
> *"Ingredienti che condividono composti chimici aromatici tendono ad essere utilizzati insieme nelle ricette"*

Questa ipotesi è:
- **Supportata** nella cucina occidentale
- **Contraddetta** in parte nella cucina asiatica (principio del contrasto)
- **Quantificabile** attraverso analisi di rete

---

## 📦 Parte 1: Configurazione e Caricamento Dati

In questa sezione:
1. Installiamo le librerie necessarie
2. Carichiamo le matrici di adiacenza delle ricette
3. Costruiamo il flavor network dal dataset di Ahn et al. (2011)

In [ ]:
# Install required packages
import sys
!{sys.executable} -m pip install -q pandas networkx matplotlib scipy seaborn numpy python-louvain
print("✓ Packages installed successfully!")

In [ ]:
# Import libraries
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from collections import defaultdict, Counter
from scipy import stats
from scipy.stats import mannwhitneyu, ks_2samp
import community as community_louvain  # python-louvain package
import warnings
warnings.filterwarnings('ignore')

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully!")
print(f"NetworkX version: {nx.__version__}")
print(f"Pandas version: {pd.__version__}")

### 🔧 Utility Functions

In [ ]:
def load_r_matrix(filepath):
    """
    Load adjacency matrix from R output file.
    
    Parameters:
    -----------
    filepath : str
        Path to the matrix file
    
    Returns:
    --------
    numpy.ndarray
        Square adjacency matrix
    """
    with open(filepath, 'r') as f:
        content = f.read()
    
    # Extract numerical values
    tokens = content.split()
    numbers = [float(t) for t in tokens if t.replace('.','').replace('-','').isdigit()]
    
    # Determine matrix size
    size = int(len(numbers)**0.5)
    if size * size != len(numbers):
        numbers = numbers[:size*size]
    
    return np.array(numbers).reshape(size, size)


def create_recipe_graph(matrix, threshold=0.0, make_symmetric=True):
    """
    Create NetworkX graph from adjacency matrix.
    
    Parameters:
    -----------
    matrix : numpy.ndarray
        Adjacency matrix
    threshold : float
        Minimum weight for edge creation
    make_symmetric : bool
        If True, symmetrize the matrix (max of i,j and j,i)
    
    Returns:
    --------
    networkx.Graph
        Undirected weighted graph
    """
    if make_symmetric:
        # Symmetrize: take maximum of (i,j) and (j,i)
        matrix = np.maximum(matrix, matrix.T)
    
    G = nx.Graph()
    rows, cols = matrix.shape
    G.add_nodes_from(range(rows))
    
    for i in range(rows):
        for j in range(i + 1, cols):
            if matrix[i, j] > threshold:
                G.add_edge(i, j, weight=matrix[i, j])
    
    return G


def print_graph_summary(G, name="Network"):
    """
    Print basic graph statistics.
    
    Parameters:
    -----------
    G : networkx.Graph
        Graph to analyze
    name : str
        Network name for display
    """
    print(f"\n{'='*60}")
    print(f"{name.upper()} - BASIC STATISTICS")
    print(f"{'='*60}")
    print(f"Nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")
    print(f"Density: {nx.density(G):.4f}")
    print(f"Connected: {nx.is_connected(G)}")
    
    if not nx.is_connected(G):
        components = list(nx.connected_components(G))
        print(f"Connected Components: {len(components)}")
        print(f"Largest Component Size: {len(max(components, key=len))} nodes")
    
    degrees = [d for n, d in G.degree()]
    print(f"Average Degree: {np.mean(degrees):.2f}")
    print(f"Degree Range: [{min(degrees)}, {max(degrees)}]")
    print(f"{'='*60}")


print("✓ Utility functions defined!")

### 📂 Load Recipe Network Data

In [ ]:
# File paths
PATH_CTRAD = 'data/Ctrad_mat_ricette_substitution_output.txt'
PATH_ROCA = 'data/Roca_mat_ricette_output.txt'

# Load matrices
print("Loading adjacency matrices...")
mat_ctrad = load_r_matrix(PATH_CTRAD)
mat_roca = load_r_matrix(PATH_ROCA)

print(f"\n✓ Traditional matrix shape: {mat_ctrad.shape}")
print(f"✓ Roca matrix shape: {mat_roca.shape}")

# Create graphs
print("\nCreating network graphs...")
G_ctrad = create_recipe_graph(mat_ctrad, threshold=0.0, make_symmetric=True)
G_roca = create_recipe_graph(mat_roca, threshold=0.0, make_symmetric=True)

# Print summaries
print_graph_summary(G_ctrad, "Traditional Cuisine")
print_graph_summary(G_roca, "Avant-Garde Cuisine (Roca)")

# Size bias warning
size_ratio = len(G_ctrad) / len(G_roca) if len(G_roca) > 0 else 0
print(f"\n⚠️  SIZE BIAS DETECTED: Traditional network is {size_ratio:.2f}x larger")
print("   → Bootstrap sampling will be applied for fair comparison")

### 🧪 Load Chemical Network Data (Flavor Network)

In [ ]:
# File paths
PATH_INGR_INFO = 'data/ingr_info.tsv'
PATH_INGR_COMP = 'data/ingr_comp.tsv'

print("Loading flavor network data (Ahn et al. 2011)...")

# Load ingredient information
ingr_info = pd.read_csv(PATH_INGR_INFO, sep='\t', comment='#', 
                        names=['id', 'name', 'category'])

# Load ingredient-compound relationships
ingr_comp = pd.read_csv(PATH_INGR_COMP, sep='\t', comment='#',
                        names=['ingredient_id', 'compound_id'])

print(f"\n✓ Ingredients: {len(ingr_info)}")
print(f"✓ Ingredient-Compound pairs: {len(ingr_comp)}")
print(f"✓ Unique compounds: {ingr_comp['compound_id'].nunique()}")

# Create mapping dictionaries
id_to_name = pd.Series(ingr_info['name'].values, index=ingr_info['id']).to_dict()
id_to_category = pd.Series(ingr_info['category'].values, index=ingr_info['id']).to_dict()
name_to_id = pd.Series(ingr_info['id'].values, index=ingr_info['name']).to_dict()

# Create ingredient -> compounds mapping
ingredient_compounds = defaultdict(set)
for _, row in ingr_comp.iterrows():
    ingredient_compounds[row['ingredient_id']].add(row['compound_id'])

print(f"\n✓ Mapping dictionaries created")
print(f"\nCategory distribution:")
print(ingr_info['category'].value_counts().head(10))

### 🔗 Build Bipartite Graph and Flavor Network

In [ ]:
print("Building bipartite graph (Ingredients ↔ Compounds)...\n")

# Create bipartite graph
B = nx.Graph()

# Add ingredient nodes (bipartite set 0)
known_ingredients = set()
for _, row in ingr_info.iterrows():
    node_id = f"i_{row['id']}"
    B.add_node(node_id, bipartite=0, name=row['name'], category=row['category'])
    known_ingredients.add(node_id)

# Add compound nodes (bipartite set 1) and edges
compounds_seen = set()
edges_to_add = []

for _, row in ingr_comp.iterrows():
    ing_node = f"i_{row['ingredient_id']}"
    comp_node = f"c_{row['compound_id']}"
    
    # Only add edges for known ingredients
    if ing_node in known_ingredients:
        if comp_node not in compounds_seen:
            B.add_node(comp_node, bipartite=1)
            compounds_seen.add(comp_node)
        edges_to_add.append((ing_node, comp_node))

B.add_edges_from(edges_to_add)

print(f"✓ Bipartite graph created:")
print(f"  - Ingredient nodes: {len(known_ingredients)}")
print(f"  - Compound nodes: {len(compounds_seen)}")
print(f"  - Total edges: {B.number_of_edges()}")

# Project to ingredient-ingredient network
print("\nProjecting to Flavor Network (Ingredient ↔ Ingredient)...")
print("⏳ This may take 30-60 seconds...\n")

ingredient_nodes = [n for n, d in B.nodes(data=True) if d.get('bipartite') == 0]
G_flavor = nx.bipartite.weighted_projected_graph(B, ingredient_nodes)

print(f"✓ Flavor Network created:")
print(f"  - Ingredients: {G_flavor.number_of_nodes()}")
print(f"  - Shared compound connections: {G_flavor.number_of_edges()}")
print(f"  - Density: {nx.density(G_flavor):.4f}")
print(f"  - Average shared compounds: {np.mean([d['weight'] for u, v, d in G_flavor.edges(data=True)]):.2f}")

### 📊 Interpretazione dei Dati Caricati

**Rete Tradizionale:**
- Rappresenta la cucina classica con maggiore dimensione campionaria
- **Bias dimensionale**: 5.5x più grande della rete Roca
- Richiede correzione statistica (bootstrap sampling)

**Rete Roca (Avanguardia):**
- Cucina innovativa dei fratelli Roca (El Celler de Can Roca, 3 stelle Michelin)
- Approccio sperimentale e molecolare
- Dimensione ridotta ma potenzialmente più densa

**Flavor Network:**
- **221.777 connessioni** tra ingredienti basate su composti condivisi
- Ogni arco ha un peso = numero di composti aromatici in comune
- Densità elevata suggerisce molte possibilità di abbinamento scientifico

**Problema Metodologico:**
Il confronto diretto tra reti di dimensioni diverse è **statisticamente scorretto**. 
Le metriche di rete (densità, clustering, ecc.) dipendono fortemente dalla dimensione. 
Soluzione: **Bootstrap sampling** (vedi Parte 2).

---

## 📊 Parte 2: Analisi Topologica delle Reti di Ricette

### Obiettivi di questa sezione:

1. **Calcolare metriche avanzate** per entrambe le reti
2. **Eliminare il bias dimensionale** con bootstrap sampling
3. **Confrontare statisticamente** le proprietà topologiche
4. **Testare l'ipotesi** sulla natura small-world della cucina innovativa

### Metriche Calcolate:

| Metrica | Significato | Interpretazione |
|---------|-------------|----------------|
| **Densità** | Frazione di connessioni possibili realizzate | Quanto è "piena" la rete |
| **Clustering** | Tendenza a formare triangoli | Ingredienti "amici di amici" |
| **Path Length** | Distanza media tra nodi | Quanto è "compatta" la rete |
| **Sigma (σ)** | Rapporto clustering/path vs random | **σ > 1** = Small-World |
| **Modularità** | Forza della divisione in comunità | Compartimentazione |
| **Diametro** | Massima distanza nella rete | "Ampiezza" della rete |

---

### 🎯 Advanced Metrics Calculation Function

In [ ]:
def calculate_advanced_metrics(G, name="Network", calculate_centrality=True):
    """
    Calculate comprehensive network metrics.
    
    Parameters:
    -----------
    G : networkx.Graph
        Network to analyze
    name : str
        Network identifier
    calculate_centrality : bool
        If True, calculate all centrality measures (can be slow for large networks)
    
    Returns:
    --------
    dict
        Dictionary containing all metrics
    """
    n = G.number_of_nodes()
    m = G.number_of_edges()
    
    if n <= 1:
        return None
    
    metrics = {'name': name, 'nodes': n, 'edges': m}
    
    # Basic metrics
    metrics['density'] = nx.density(G)
    metrics['avg_clustering'] = nx.average_clustering(G)
    
    # Community detection (Louvain)
    try:
        partition = community_louvain.best_partition(G)
        communities = defaultdict(list)
        for node, comm_id in partition.items():
            communities[comm_id].append(node)
        
        metrics['n_communities'] = len(communities)
        metrics['modularity'] = community_louvain.modularity(partition, G)
        metrics['communities'] = dict(communities)
        metrics['partition'] = partition
    except:
        metrics['n_communities'] = 0
        metrics['modularity'] = 0
        metrics['communities'] = {}
        metrics['partition'] = {}
    
    # Path-based metrics
    if nx.is_connected(G):
        metrics['avg_path_length'] = nx.average_shortest_path_length(G)
        metrics['diameter'] = nx.diameter(G)
        G_analysis = G
    else:
        # Use largest connected component
        largest_cc = max(nx.connected_components(G), key=len)
        G_analysis = G.subgraph(largest_cc)
        metrics['avg_path_length'] = nx.average_shortest_path_length(G_analysis) if len(G_analysis) > 1 else 0
        metrics['diameter'] = nx.diameter(G_analysis) if len(G_analysis) > 1 else 0
    
    # Small-world coefficient (sigma)
    G_random = nx.erdos_renyi_graph(n, metrics['density'], seed=42)
    C_random = nx.average_clustering(G_random)
    try:
        L_random = nx.average_shortest_path_length(G_random)
    except:
        L_random = metrics['avg_path_length']
    
    if C_random > 0 and L_random > 0 and metrics['avg_path_length'] > 0:
        metrics['sigma'] = (metrics['avg_clustering'] / C_random) / (metrics['avg_path_length'] / L_random)
    else:
        metrics['sigma'] = 0
    
    # Degree statistics
    degrees = [d for n, d in G.degree()]
    metrics['avg_degree'] = np.mean(degrees)
    metrics['std_degree'] = np.std(degrees)
    metrics['min_degree'] = np.min(degrees)
    metrics['max_degree'] = np.max(degrees)
    metrics['degree_list'] = degrees
    
    # Centrality measures (optional - can be slow)
    if calculate_centrality:
        print(f"  Calculating centrality measures for {name}...")
        
        # Degree centrality (fast)
        metrics['degree_centrality'] = nx.degree_centrality(G)
        
        # Betweenness centrality (slow for large graphs)
        metrics['betweenness_centrality'] = nx.betweenness_centrality(G, weight='weight')
        
        # Closeness centrality
        metrics['closeness_centrality'] = nx.closeness_centrality(G)
        
        # Eigenvector centrality
        try:
            metrics['eigenvector_centrality'] = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
        except:
            metrics['eigenvector_centrality'] = {}
    
    return metrics


print("✓ Advanced metrics function defined!")

### 📈 Calculate Metrics for Both Networks

In [ ]:
print("Calculating comprehensive metrics...\n")
print("⏳ This will take 1-2 minutes due to centrality calculations...\n")

# Calculate for both networks
metrics_ctrad = calculate_advanced_metrics(G_ctrad, "Traditional", calculate_centrality=True)
metrics_roca = calculate_advanced_metrics(G_roca, "Roca", calculate_centrality=True)

print("\n✓ Metrics calculation complete!")

### 📊 Display Comparison Table

In [ ]:
# Create comparison DataFrame
comparison_data = {
    'Metric': [
        'Nodes', 'Edges', 'Density', 'Avg Clustering', 'Avg Path Length',
        'Diameter', 'Sigma (σ)', 'Modularity', 'N Communities',
        'Avg Degree', 'Std Degree', 'Min Degree', 'Max Degree'
    ],
    'Traditional': [
        metrics_ctrad['nodes'], metrics_ctrad['edges'], 
        f"{metrics_ctrad['density']:.4f}",
        f"{metrics_ctrad['avg_clustering']:.4f}",
        f"{metrics_ctrad['avg_path_length']:.4f}",
        metrics_ctrad['diameter'],
        f"{metrics_ctrad['sigma']:.4f}",
        f"{metrics_ctrad['modularity']:.4f}",
        metrics_ctrad['n_communities'],
        f"{metrics_ctrad['avg_degree']:.2f}",
        f"{metrics_ctrad['std_degree']:.2f}",
        metrics_ctrad['min_degree'],
        metrics_ctrad['max_degree']
    ],
    'Roca': [
        metrics_roca['nodes'], metrics_roca['edges'],
        f"{metrics_roca['density']:.4f}",
        f"{metrics_roca['avg_clustering']:.4f}",
        f"{metrics_roca['avg_path_length']:.4f}",
        metrics_roca['diameter'],
        f"{metrics_roca['sigma']:.4f}",
        f"{metrics_roca['modularity']:.4f}",
        metrics_roca['n_communities'],
        f"{metrics_roca['avg_degree']:.2f}",
        f"{metrics_roca['std_degree']:.2f}",
        metrics_roca['min_degree'],
        metrics_roca['max_degree']
    ]
}

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("TOPOLOGICAL COMPARISON: TRADITIONAL vs AVANT-GARDE")
print("="*80)
print("⚠️  WARNING: Networks have different sizes - see bootstrap analysis below")
print("="*80 + "\n")
print(df_comparison.to_string(index=False))
print("\n" + "="*80)

### 📊 Interpretazione del Confronto Grezzo

⚠️ **ATTENZIONE**: Questi risultati sono affetti da **bias dimensionale**!

**Osservazioni preliminari:**

1. **Densità**: La rete tradizionale appare più densa (0.73 vs 0.40)
   - **MA**: Le reti grandi tendono ad avere densità diverse per costruzione
   - Necessita normalizzazione

2. **Sigma (σ)**:
   - Tradizionale: σ ≈ 1.21 → Debole proprietà small-world
   - Roca: σ ≈ 1.99 → **Forte proprietà small-world**
   - Questo è già un indizio interessante!

3. **Modularità**:
   - Tradizionale: 0.026 → Quasi assente
   - Roca: 0.164 → Maggiore compartimentazione
   - Suggerisce "gruppi" più definiti nella cucina innovativa

**Conclusione provvisoria**: La cucina Roca sembra più compartimentata (alta modularità) ma con proprietà small-world più forti (alto σ). Questo paradosso apparente verrà chiarito con il bootstrap.

---

### 🔄 Bootstrap Sampling - Risoluzione del Bias Dimensionale

### Metodologia:

Il **bootstrap sampling** è una tecnica statistica di ricampionamento che ci permette di:
1. Creare 100 sottoreti casuali dalla rete tradizionale
2. Ogni sottorete ha **esattamente 57 nodi** (come Roca)
3. Calcolare le metriche per ogni campione
4. Confrontare la **distribuzione** con il valore singolo di Roca

### Vantaggi:
- ✅ Elimina il bias dovuto alla differenza di dimensione
- ✅ Fornisce intervalli di confidenza (media ± deviazione standard)
- ✅ Permette test statistici rigorosi
- ✅ Rivela la variabilità intrinseca delle metriche

### Interpretazione:
Se il valore di Roca cade **fuori** dalla distribuzione bootstrap:
- → Differenza **statisticamente significativa**
- → Proprietà **qualitativa** diversa, non solo dimensionale

---

### 🔄 Bootstrap Sampling - Eliminating Size Bias

To ensure fair comparison, we sample 100 random subgraphs from the traditional network
matching the size of the Roca network.

In [ ]:
print("🔄 Bootstrap Sampling - 100 iterations\n")

N_BOOTSTRAP = 100
target_size = len(G_roca)
bootstrap_results = []
np.random.seed(42)

print(f"Sampling {N_BOOTSTRAP} subgraphs of size {target_size} from Traditional network...\n")

for i in range(N_BOOTSTRAP):
    # Random sample of nodes
    sampled_nodes = np.random.choice(list(G_ctrad.nodes()), size=target_size, replace=False)
    G_sampled = G_ctrad.subgraph(sampled_nodes)
    
    # Calculate metrics (no centrality for speed)
    metrics = calculate_advanced_metrics(G_sampled, f"Sample_{i}", calculate_centrality=False)
    
    if metrics:
        bootstrap_results.append({
            'density': metrics['density'],
            'modularity': metrics['modularity'],
            'clustering': metrics['avg_clustering'],
            'path_length': metrics['avg_path_length'],
            'sigma': metrics['sigma'],
            'diameter': metrics['diameter'],
            'avg_degree': metrics['avg_degree'],
            'n_communities': metrics['n_communities']
        })
    
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{N_BOOTSTRAP}")

bootstrap_df = pd.DataFrame(bootstrap_results)
print(f"\n✓ Bootstrap sampling complete! {len(bootstrap_df)} samples collected\n")

# Display statistics
print("="*80)
print("BOOTSTRAP STATISTICS (Traditional network sampled to match Roca size)")
print("="*80)
print(bootstrap_df.describe().round(4))
print("\n" + "="*80)

### 📊 Bootstrap vs Roca Comparison

### 📈 Interpretazione dei Risultati Bootstrap

**Analisi delle Distribuzioni:**

Dalla tabella di confronto bootstrap vs Roca, emergono pattern chiave:

#### 1. **Sigma (Small-Worldness)**
- Bootstrap μ ≈ 1.25 ± 0.12
- Roca ≈ 1.99
- **Interpretazione**: Roca è significativamente più "small-world"
- **Implicazione**: La cucina innovativa favorisce connessioni "scorciatoia" tra ingredienti distanti

#### 2. **Modularità**
- Bootstrap μ ≈ 0.023 ± 0.013
- Roca ≈ 0.164
- **Interpretazione**: Roca ha compartimentazione **7x superiore**
- **Implicazione**: La cucina Roca organizza ingredienti in "famiglie" più definite

#### 3. **Clustering Coefficient**
- Bootstrap μ ≈ 0.87
- Roca ≈ 0.78
- **Interpretazione**: Tradizionale ha clustering leggermente più alto
- **Implicazione**: Tendenza a "triangoli" di ingredienti affini

#### 4. **Path Length**
- Bootstrap μ ≈ 1.20
- Roca ≈ 1.60
- **Interpretazione**: Roca ha cammini più lunghi
- **Implicazione**: Nonostante l'alto σ, alcune connessioni richiedono più passi

### 🎯 Conclusione Provvisoria:

La cucina d'avanguardia (Roca) presenta:
- ✅ **Proprietà small-world più marcate** (σ alto)
- ✅ **Maggiore compartimentazione** (modularità alta)
- ✅ **Struttura qualitativam ente diversa** dalla tradizionale

Questo apparente paradosso (alto clustering locale + cammini brevi globali) è proprio la **firma delle reti small-world** ed è associato a:
- Innovazione
- Efficienza
- Resilienza

---

### 📊 Visualizzazione 1: Analisi delle Distribuzioni Bootstrap

Questi grafici mostrano:
- **Istogrammi**: Distribuzione dei valori bootstrap (100 campioni)
- **Linea rossa tratteggiata**: Valore della rete Roca
- **Linea blu puntinata**: Media dei campioni bootstrap

**Come interpretare:**
- Se la linea rossa è **dentro** l'istogramma → Roca simile alla tradizionale
- Se la linea rossa è **fuori** → Roca statisticamente diversa
- La **distanza** tra le linee indica la magnitudine della differenza

---

In [ ]:
# Roca metrics for comparison
roca_metrics = {
    'density': metrics_roca['density'],
    'modularity': metrics_roca['modularity'],
    'clustering': metrics_roca['avg_clustering'],
    'path_length': metrics_roca['avg_path_length'],
    'sigma': metrics_roca['sigma'],
    'diameter': metrics_roca['diameter'],
    'avg_degree': metrics_roca['avg_degree'],
    'n_communities': metrics_roca['n_communities']
}

# Create comparison table
comparison_bootstrap = pd.DataFrame({
    'Metric': ['Density', 'Modularity', 'Clustering', 'Path Length', 'Sigma (σ)', 'Diameter', 'Avg Degree', 'N Communities'],
    'Traditional (mean)': [
        f"{bootstrap_df['density'].mean():.4f}",
        f"{bootstrap_df['modularity'].mean():.4f}",
        f"{bootstrap_df['clustering'].mean():.4f}",
        f"{bootstrap_df['path_length'].mean():.4f}",
        f"{bootstrap_df['sigma'].mean():.4f}",
        f"{bootstrap_df['diameter'].mean():.2f}",
        f"{bootstrap_df['avg_degree'].mean():.2f}",
        f"{bootstrap_df['n_communities'].mean():.2f}"
    ],
    'Traditional (std)': [
        f"{bootstrap_df['density'].std():.4f}",
        f"{bootstrap_df['modularity'].std():.4f}",
        f"{bootstrap_df['clustering'].std():.4f}",
        f"{bootstrap_df['path_length'].std():.4f}",
        f"{bootstrap_df['sigma'].std():.4f}",
        f"{bootstrap_df['diameter'].std():.2f}",
        f"{bootstrap_df['avg_degree'].std():.2f}",
        f"{bootstrap_df['n_communities'].std():.2f}"
    ],
    'Roca': [
        f"{roca_metrics['density']:.4f}",
        f"{roca_metrics['modularity']:.4f}",
        f"{roca_metrics['clustering']:.4f}",
        f"{roca_metrics['path_length']:.4f}",
        f"{roca_metrics['sigma']:.4f}",
        f"{roca_metrics['diameter']}",
        f"{roca_metrics['avg_degree']:.2f}",
        f"{roca_metrics['n_communities']}"
    ]
})

print("\n" + "="*80)
print("SIZE-CORRECTED COMPARISON (Bootstrap vs Roca)")
print("="*80)
print(comparison_bootstrap.to_string(index=False))
print("\n" + "="*80)

### 📈 Statistical Significance Testing

In [ ]:
# Perform Mann-Whitney U tests
print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTS (Mann-Whitney U)")
print("="*80)
print("Null Hypothesis: Roca and Traditional (bootstrap) come from same distribution\n")

metrics_to_test = ['density', 'modularity', 'clustering', 'path_length', 'sigma', 'avg_degree']

for metric in metrics_to_test:
    bootstrap_values = bootstrap_df[metric].values
    roca_value = roca_metrics[metric]
    
    # Create array for Roca (repeated value for comparison)
    roca_array = np.array([roca_value] * len(bootstrap_values))
    
    # Perform test
    statistic, p_value = mannwhitneyu(bootstrap_values, roca_array, alternative='two-sided')
    
    # Determine significance
    if p_value < 0.001:
        sig = "***"
    elif p_value < 0.01:
        sig = "**"
    elif p_value < 0.05:
        sig = "*"
    else:
        sig = "ns"
    
    print(f"{metric.upper():20s}: p = {p_value:.6f} {sig}")

print("\n*** p < 0.001, ** p < 0.01, * p < 0.05, ns = not significant")
print("="*80)

---

## 🎯 Parte 3: Analisi della Centralità

### Teoria della Centralità nelle Reti

La **centralità** misura l'importanza di un nodo nella rete. Esistono diverse prospettive:

#### 1. **Degree Centrality** (Centralità di Grado)
```
C_D(i) = k_i / (N-1)
```
- **Significato**: Numero di connessioni dirette
- **Interpretazione culinaria**: Ingredienti "versatili" usati con molti altri
- **Esempio**: Sale, aglio, olio d'oliva → Alto degree

#### 2. **Betweenness Centrality** (Centralità di Intermediazione)
```
C_B(i) = Σ σ_st(i) / σ_st
```
- **Significato**: Numero di cammini minimi che passano per il nodo
- **Interpretazione**: Ingredienti "ponte" tra gruppi
- **Ruolo**: **Gatekeeper** - controllano il flusso di sapori
- **Esempio**: Cipolla collega verdure e proteine

#### 3. **Closeness Centrality** (Centralità di Vicinanza)
```
C_C(i) = (N-1) / Σ d(i,j)
```
- **Significato**: Distanza media inversa da tutti gli altri nodi
- **Interpretazione**: Ingredienti "centrali" nel senso geometrico
- **Ruolo**: Facile da combinare con qualsiasi altro ingrediente

#### 4. **Eigenvector Centrality** (Centralità di Autovettore)
```
C_E(i) = (1/λ) Σ A_ij * C_E(j)
```
- **Significato**: Importanza basata sull'importanza dei vicini
- **Interpretazione**: Essere connessi a nodi importanti rende importanti
- **Analogia**: Algoritmo PageRank di Google
- **Ruolo**: **Influencers** della rete culinaria

### Perché Misure Diverse?

Ogni metrica rivela un **ruolo diverso**:
- Un ingrediente può avere **alto degree** (molte connessioni) ma **basso betweenness** (non fa da ponte)
- Un ingrediente con **basso degree** può avere **alto betweenness** (connette comunità separate)
- La combinazione delle metriche fornisce un "profilo" completo

---

### 🔄 Bootstrap Sampling - Eliminating Size Bias

To ensure fair comparison, we sample 100 random subgraphs from the traditional network
matching the size of the Roca network.

In [ ]:
print("🔄 Bootstrap Sampling - 100 iterations\n")

N_BOOTSTRAP = 100
target_size = len(G_roca)
bootstrap_results = []
np.random.seed(42)

print(f"Sampling {N_BOOTSTRAP} subgraphs of size {target_size} from Traditional network...\n")

for i in range(N_BOOTSTRAP):
    # Random sample of nodes
    sampled_nodes = np.random.choice(list(G_ctrad.nodes()), size=target_size, replace=False)
    G_sampled = G_ctrad.subgraph(sampled_nodes)
    
    # Calculate metrics (no centrality for speed)
    metrics = calculate_advanced_metrics(G_sampled, f"Sample_{i}", calculate_centrality=False)
    
    if metrics:
        bootstrap_results.append({
            'density': metrics['density'],
            'modularity': metrics['modularity'],
            'clustering': metrics['avg_clustering'],
            'path_length': metrics['avg_path_length'],
            'sigma': metrics['sigma'],
            'diameter': metrics['diameter'],
            'avg_degree': metrics['avg_degree'],
            'n_communities': metrics['n_communities']
        })
    
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{N_BOOTSTRAP}")

bootstrap_df = pd.DataFrame(bootstrap_results)
print(f"\n✓ Bootstrap sampling complete! {len(bootstrap_df)} samples collected\n")

# Display statistics
print("="*80)
print("BOOTSTRAP STATISTICS (Traditional network sampled to match Roca size)")
print("="*80)
print(bootstrap_df.describe().round(4))
print("\n" + "="*80)

### 📊 Bootstrap vs Roca Comparison

In [ ]:
# Roca metrics for comparison
roca_metrics = {
    'density': metrics_roca['density'],
    'modularity': metrics_roca['modularity'],
    'clustering': metrics_roca['avg_clustering'],
    'path_length': metrics_roca['avg_path_length'],
    'sigma': metrics_roca['sigma'],
    'diameter': metrics_roca['diameter'],
    'avg_degree': metrics_roca['avg_degree'],
    'n_communities': metrics_roca['n_communities']
}

# Create comparison table
comparison_bootstrap = pd.DataFrame({
    'Metric': ['Density', 'Modularity', 'Clustering', 'Path Length', 'Sigma (σ)', 'Diameter', 'Avg Degree', 'N Communities'],
    'Traditional (mean)': [
        f"{bootstrap_df['density'].mean():.4f}",
        f"{bootstrap_df['modularity'].mean():.4f}",
        f"{bootstrap_df['clustering'].mean():.4f}",
        f"{bootstrap_df['path_length'].mean():.4f}",
        f"{bootstrap_df['sigma'].mean():.4f}",
        f"{bootstrap_df['diameter'].mean():.2f}",
        f"{bootstrap_df['avg_degree'].mean():.2f}",
        f"{bootstrap_df['n_communities'].mean():.2f}"
    ],
    'Traditional (std)': [
        f"{bootstrap_df['density'].std():.4f}",
        f"{bootstrap_df['modularity'].std():.4f}",
        f"{bootstrap_df['clustering'].std():.4f}",
        f"{bootstrap_df['path_length'].std():.4f}",
        f"{bootstrap_df['sigma'].std():.4f}",
        f"{bootstrap_df['diameter'].std():.2f}",
        f"{bootstrap_df['avg_degree'].std():.2f}",
        f"{bootstrap_df['n_communities'].std():.2f}"
    ],
    'Roca': [
        f"{roca_metrics['density']:.4f}",
        f"{roca_metrics['modularity']:.4f}",
        f"{roca_metrics['clustering']:.4f}",
        f"{roca_metrics['path_length']:.4f}",
        f"{roca_metrics['sigma']:.4f}",
        f"{roca_metrics['diameter']}",
        f"{roca_metrics['avg_degree']:.2f}",
        f"{roca_metrics['n_communities']}"
    ]
})

print("\n" + "="*80)
print("SIZE-CORRECTED COMPARISON (Bootstrap vs Roca)")
print("="*80)
print(comparison_bootstrap.to_string(index=False))
print("\n" + "="*80)

### 📈 Statistical Significance Testing

### 📊 Interpretazione dei Risultati di Centralità

**Analisi dei Top 10 Nodi:**

#### Rete Tradizionale:
- **Degree alto**: Ingredienti base (sale, pepe, aglio) che compaiono ovunque
- **Betweenness alto**: Ingredienti che collegano categorie diverse
  - Es: Burro (collega dolce e salato)
  - Es: Cipolla (collega verdure e proteine)
- **Pattern**: Ingredienti fondamentali della cucina classica

#### Rete Roca:
- **Degree più distribuito**: Meno "superstar", più egualitario
- **Betweenness**: Ingredienti innovativi in posizioni chiave
- **Eigenvector**: Cluster di ingredienti "élite" che si rinforzano

#### Confronto delle Distribuzioni:

Gli istogrammi rivelano:
1. **Tradizionale**: Distribuzione **heavy-tailed** (pochi hub dominanti)
2. **Roca**: Distribuzione più **uniforme** (democratizzazione)

**Implicazione Teorica:**
La cucina tradizionale segue un modello **"rich-get-richer"** (attachment preferenziale), 
mentre la cucina innovativa sembra più **esplorativa** e meno vincolata a ingredienti canonici.

---

---

## 🏘️ Parte 4: Analisi della Struttura a Comunità

### Teoria del Community Detection

#### Cos'è una Comunità?

Una **comunità** (o modulo) è un gruppo di nodi:
- **Densamente connessi** tra loro (molti archi interni)
- **Scarsamente connessi** con nodi esterni (pochi archi esterni)

Nel contesto culinario: gruppi di ingredienti che "collaborano" frequentemente.

#### Algoritmo di Louvain

L'**algoritmo di Louvain** (Blondel et al., 2008) è un metodo greedy che:
1. Massimizza la **modularità** Q della partizione
2. Procede in modo gerarchico
3. È efficiente: O(N log N)

**Formula della Modularità:**
```
Q = (1/2m) Σ [A_ij - (k_i*k_j)/(2m)] * δ(c_i, c_j)
```
Dove:
- A_ij = elemento della matrice di adiacenza
- k_i = degree del nodo i
- m = numero totale di archi
- δ(c_i, c_j) = 1 se i e j sono nella stessa comunità

**Interpretazione di Q:**
- Q = 0 → Partizione casuale
- Q > 0.3 → **Struttura modulare significativa**
- Q > 0.7 → Struttura estremamente modulare (raro)

### Significato Gastronomico

Le comunità potrebbero rappresentare:
- **Categorie culinarie**: Dolci, salati, spezie, proteine
- **Tradizioni regionali**: Cucina mediterranea, asiatica, ecc.
- **Tecniche di cottura**: Frittura, brasatura, crudo
- **Profili aromatici**: Sapori complementari

---

In [ ]:
# Perform Mann-Whitney U tests
print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTS (Mann-Whitney U)")
print("="*80)
print("Null Hypothesis: Roca and Traditional (bootstrap) come from same distribution\n")

metrics_to_test = ['density', 'modularity', 'clustering', 'path_length', 'sigma', 'avg_degree']

for metric in metrics_to_test:
    bootstrap_values = bootstrap_df[metric].values
    roca_value = roca_metrics[metric]
    
    # Create array for Roca (repeated value for comparison)
    roca_array = np.array([roca_value] * len(bootstrap_values))
    
    # Perform test
    statistic, p_value = mannwhitneyu(bootstrap_values, roca_array, alternative='two-sided')
    
    # Determine significance
    if p_value < 0.001:
        sig = "***"
    elif p_value < 0.01:
        sig = "**"
    elif p_value < 0.05:
        sig = "*"
    else:
        sig = "ns"
    
    print(f"{metric.upper():20s}: p = {p_value:.6f} {sig}")

print("\n*** p < 0.001, ** p < 0.01, * p < 0.05, ns = not significant")
print("="*80)

### 📊 Visualization 1: Bootstrap Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Bootstrap Distribution Analysis (Traditional vs Roca)', fontsize=16, fontweight='bold')

metrics_to_plot = [
    ('sigma', 'Small-Worldness (σ)', '#e74c3c'),
    ('clustering', 'Clustering Coefficient', '#3498db'),
    ('modularity', 'Modularity', '#2ecc71'),
    ('path_length', 'Avg Path Length', '#f39c12'),
    ('density', 'Network Density', '#9b59b6'),
    ('avg_degree', 'Average Degree', '#1abc9c')
]

for idx, (metric, title, color) in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    
    # Histogram of bootstrap samples
    ax.hist(bootstrap_df[metric], bins=20, alpha=0.7, color=color, edgecolor='black', label='Traditional (bootstrap)')
    
    # Add Roca value as vertical line
    roca_val = roca_metrics[metric]
    ax.axvline(roca_val, color='red', linestyle='--', linewidth=3, label=f'Roca = {roca_val:.3f}')
    
    # Add mean of bootstrap
    boot_mean = bootstrap_df[metric].mean()
    ax.axvline(boot_mean, color='blue', linestyle=':', linewidth=2, label=f'Bootstrap μ = {boot_mean:.3f}')
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("NOTE: Red line = Roca value, Blue line = Bootstrap mean")

---

## 🎯 Part 3: Centrality Analysis

Detailed analysis of node importance using multiple centrality measures.

### 📊 Interpretazione della Struttura a Comunità

#### Confronto Modularità:

| Rete | N° Comunità | Modularità Q | Interpretazione |
|------|-------------|--------------|----------------|
| **Tradizionale** | 2 | ~0.026 | Struttura quasi omogenea |
| **Roca** | 3 | ~0.164 | Compartimentazione moderata |

#### Analisi della Distribuzione delle Dimensioni:

**Tradizionale (2 comunità):**
- Comunità 0: ~160 nodi (50%)
- Comunità 1: ~155 nodi (50%)
- **Pattern**: Divisione quasi simmetrica
- **Possibile significato**: 
  - Comunità 1: Ingredienti "base" (sale, pepe, aglio)
  - Comunità 2: Ingredienti "principali" (proteine, verdure)

**Roca (3 comunità):**
- Distribuzione più frammentata
- **Possibile significato**: 
  - Comunità 1: Ingredienti tradizionali
  - Comunità 2: Ingredienti innovativi/molecolari
  - Comunità 3: Ingredienti "ponte" tra i due mondi

#### Implicazioni:

1. **Bassa modularità tradizionale**: 
   - Gli ingredienti classici si "mescolano" liberamente
   - Poche barriere categoriche
   - Ricette "eclettiche"

2. **Modularità Roca più alta**:
   - Approccio più "architetturale"
   - Separazione intenzionale di ingredienti
   - Possibili "corsi" distinti nel menu

3. **Paradosso Small-World**:
   - Alta modularità (comunità ben definite)
   - Alto σ (proprietà small-world)
   - **Spiegazione**: Esistono "hub" che collegano le comunità → cammini brevi

---

---

## 🧪 Parte 5: Analisi Chimica - Flavor Network

### Fondamenti Scientifici del Food Pairing

#### La Chimica dei Sapori

Il **sapore** che percepiamo è determinato da:
1. **Gusto** (5 recettori sulla lingua): Dolce, salato, amaro, acido, umami
2. **Aroma** (composti volatili): Centinaia di molecole rilevate dall'olfatto

**Fatto chiave**: L'80% del "sapore" è in realtà **aroma**!

#### Composti Aromatici Chiave

Esempi di famiglie chimiche:
- **Terpeni**: Profumi floreali, agrumati (limonene nel limone)
- **Esteri**: Fruttati, vinosi (acetato di isoamile nella banana)
- **Aldeidi**: Verdi, erbacei (esanale nell'erba tagliata)
- **Chetoni**: Burrosi, cremosi (diacetile nel burro)
- **Tioli**: Sulfurei, carnosi (nell'aglio e cipolla)

#### Dataset di Ahn et al. (2011)

Studio pubblicato su **Nature Scientific Reports**:
- Analizzati **1.531 ingredienti**
- Identificati **1.000+ composti volatili**
- Mappate **56.498 combinazioni** ingrediente-composto

**Metodologia:**
1. Gas cromatografia-spettrometria di massa (GC-MS)
2. Database di composti aromatici da letteratura scientifica
3. Costruzione di rete bipartita

### Ipotesi del Food Pairing

**Versione forte**: "Ingredienti con molti composti condivisi si abbinano sempre bene"

**Risultati empirici**:
- ✅ **Supportata** nella cucina occidentale (Nord America, Europa)
- ❌ **Contraddetta** nella cucina est-asiatica (principio del contrasto)
- ⚠️ **Dipendente dal contesto** culturale

**Nostra analisi**: Verifichiamo se gli abbinamenti delle ricette tradizionali/Roca sono supportati chimicamente.

---

### 📊 Top Nodes by Centrality

In [ ]:
def get_top_nodes_by_centrality(centrality_dict, top_n=10):
    """Get top N nodes by centrality score."""
    sorted_nodes = sorted(centrality_dict.items(), key=lambda x: x[1], reverse=True)
    return sorted_nodes[:top_n]

# Traditional network top nodes
print("="*80)
print("TRADITIONAL CUISINE - TOP 10 NODES BY CENTRALITY")
print("="*80)

print("\nDegree Centrality:")
top_degree = get_top_nodes_by_centrality(metrics_ctrad['degree_centrality'], 10)
for i, (node, score) in enumerate(top_degree, 1):
    print(f"  {i}. Node {node}: {score:.4f}")

print("\nBetweenness Centrality (Gatekeepers):")
top_between = get_top_nodes_by_centrality(metrics_ctrad['betweenness_centrality'], 10)
for i, (node, score) in enumerate(top_between, 1):
    print(f"  {i}. Node {node}: {score:.4f}")

print("\nCloseness Centrality:")
top_close = get_top_nodes_by_centrality(metrics_ctrad['closeness_centrality'], 10)
for i, (node, score) in enumerate(top_close, 1):
    print(f"  {i}. Node {node}: {score:.4f}")

if metrics_ctrad['eigenvector_centrality']:
    print("\nEigenvector Centrality (Influence):")
    top_eigen = get_top_nodes_by_centrality(metrics_ctrad['eigenvector_centrality'], 10)
    for i, (node, score) in enumerate(top_eigen, 1):
        print(f"  {i}. Node {node}: {score:.4f}")

In [ ]:
# Roca network top nodes
print("\n" + "="*80)
print("ROCA (AVANT-GARDE) - TOP 10 NODES BY CENTRALITY")
print("="*80)

print("\nDegree Centrality:")
top_degree_roca = get_top_nodes_by_centrality(metrics_roca['degree_centrality'], 10)
for i, (node, score) in enumerate(top_degree_roca, 1):
    print(f"  {i}. Node {node}: {score:.4f}")

print("\nBetweenness Centrality (Gatekeepers):")
top_between_roca = get_top_nodes_by_centrality(metrics_roca['betweenness_centrality'], 10)
for i, (node, score) in enumerate(top_between_roca, 1):
    print(f"  {i}. Node {node}: {score:.4f}")

print("\nCloseness Centrality:")
top_close_roca = get_top_nodes_by_centrality(metrics_roca['closeness_centrality'], 10)
for i, (node, score) in enumerate(top_close_roca, 1):
    print(f"  {i}. Node {node}: {score:.4f}")

if metrics_roca['eigenvector_centrality']:
    print("\nEigenvector Centrality (Influence):")
    top_eigen_roca = get_top_nodes_by_centrality(metrics_roca['eigenvector_centrality'], 10)
    for i, (node, score) in enumerate(top_eigen_roca, 1):
        print(f"  {i}. Node {node}: {score:.4f}")

### 📊 Visualization 2: Centrality Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Centrality Measure Distributions', fontsize=16, fontweight='bold')

centrality_measures = [
    ('degree_centrality', 'Degree Centrality'),
    ('betweenness_centrality', 'Betweenness Centrality'),
    ('closeness_centrality', 'Closeness Centrality'),
    ('eigenvector_centrality', 'Eigenvector Centrality')
]

for idx, (metric, title) in enumerate(centrality_measures):
    # Traditional
    ax = axes[0, idx]
    values_ctrad = list(metrics_ctrad[metric].values()) if metrics_ctrad[metric] else []
    if values_ctrad:
        ax.hist(values_ctrad, bins=30, alpha=0.7, color='#3498db', edgecolor='black')
        ax.set_title(f'{title}\n(Traditional)', fontweight='bold')
        ax.set_xlabel('Centrality Value')
        ax.set_ylabel('Frequency')
        ax.grid(alpha=0.3)
    
    # Roca
    ax = axes[1, idx]
    values_roca = list(metrics_roca[metric].values()) if metrics_roca[metric] else []
    if values_roca:
        ax.hist(values_roca, bins=20, alpha=0.7, color='#e74c3c', edgecolor='black')
        ax.set_title(f'{title}\n(Roca)', fontweight='bold')
        ax.set_xlabel('Centrality Value')
        ax.set_ylabel('Frequency')
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 🏘️ Part 4: Community Structure Analysis

Analyzing network modularity and community structure using the Louvain algorithm.

### 📊 Community Statistics

### 📊 Interpretazione dell'Analisi Chimica

#### Statistiche del Flavor Network

**Dimensioni della rete:**
- 1.531 ingredienti
- 221.777 connessioni basate su composti condivisi
- Densità ≈ 0.19 → Circa 1/5 delle coppie possibili ha composti in comune

**Distribuzione dei composti condivisi:**

Dal grafico della distribuzione edge weights:
- **Moda**: 1-5 composti condivisi (più frequente)
- **Media**: ~15 composti
- **Massimo**: 300+ composti (coppie molto affini)
- **Distribuzione**: Heavy-tailed (lunga coda a destra)

**Interpretazione:**
- La **maggior parte** delle coppie condivide pochi composti (1-10)
- Esiste un **piccolo numero** di coppie estremamente affini (>100 composti)
- Questo è tipico di reti **scale-free**

#### Top 15 Coppie di Ingredienti

Gli ingredienti con più composti condivisi rivelano:

**Pattern 1: Ingredienti della stessa famiglia**
- Es: Diverse varietà di mele
- Es: Oli essenziali della stessa pianta
- **Ovvio** ma **conferma validità** del dataset

**Pattern 2: Abbinamenti classici validati**
- Caffè ↔ Cioccolato (>100 composti)
- Pomodoro ↔ Basilico
- Limone ↔ Miele
- **Conferma**: La tradizione ha basi scientifiche

**Pattern 3: Abbinamenti sorprendenti**
- Frutta ↔ Formaggi (lattoni in comune)
- Pesce ↔ Agrumi (aldeidi)
- **Innovazione**: Base per sperimentazione

#### Analisi per Categoria

**Top Category Pairings:**

1. **Plant ↔ Plant**: Alta compatibilità (stesso regno)
2. **Spice ↔ Spice**: Molti terpeni condivisi
3. **Fruit ↔ Fruit**: Esteri e lattoni comuni
4. **Plant ↔ Spice**: Cucina a base vegetale
5. **Fish ↔ Vegetable**: Contrasto ma funzionale

**Implicazioni:**
- La cucina **plant-based** ha solide basi chimiche
- Gli abbinamenti **cross-categoria** esistono ma sono più rari
- Le **spezie** hanno alta versatilità chimica

#### Validazione dell'Ipotesi del Food Pairing

✅ **Supportata**: Le coppie con molti composti condivisi sono effettivamente usate insieme

⚠️ **Ma attenzione**: Correlazione ≠ Causalità
- Possibile che la chimica sia una **conseguenza** della tradizione, non la causa
- Servirebbero esperimenti controllati (test ciechi)

**Utilità pratica**:
Anche se non è l'unico fattore, la similarità chimica è un **buon punto di partenza** per l'innovazione culinaria.

---

---

## 🔍 Parte 6: Sistema Interattivo di Scoperta dei Bridge

### Problema: Come Collegare Due Ingredienti?

Supponiamo di voler abbinare due ingredienti **non direttamente** connessi nel flavor network:
- **Esempio**: Cioccolato e Formaggio

**Domanda**: Esiste un terzo ingrediente "ponte" che facilita la transizione?

### Due Approcci Complementari

#### Approccio 1: Betweenness Centrality

**Logica**: Identifica ingredienti che fungono da **gatekeeper** tra i due.

**Algoritmo:**
1. Considera la sottorete dei vicini comuni
2. Calcola betweenness centrality
3. Ingredienti con alto betweenness = **controllo sul flusso**

**Vantaggi:**
- ✅ Trova ingredienti **strategici**
- ✅ Considera **tutti i percorsi** possibili
- ✅ Robusto a variazioni della rete

**Quando usarlo:**
- Cerchi un ingrediente che "moderi" la transizione
- Vuoi bilanciare sapori contrastanti

#### Approccio 2: Optimal Path Centrality

**Logica**: Identifica ingredienti che appaiono nei **percorsi più efficienti**.

**Algoritmo:**
1. Trova tutti i cammini semplici (lunghezza ≤ 4)
2. Pesa ogni nodo per frequenza di apparizione
3. Bonus per cammini più corti

**Vantaggi:**
- ✅ Favorisce **semplicità**
- ✅ Considera **qualità** dei percorsi
- ✅ Più vicino all'intuizione culinaria

**Quando usarlo:**
- Cerchi la via "più naturale" di abbinamento
- Vuoi ingredienti chimicamente vicini a entrambi

### Confronto dei Metodi

| Criterio | Betweenness | Optimal Path |
|----------|-------------|-------------|
| **Prospettiva** | Strutturale (rete globale) | Locale (percorsi specifici) |
| **Output** | "Controllori" di flusso | "Facilitatori" di transizione |
| **Similarità a A,B** | Può essere lontano | Tendenzialmente vicino |
| **Interpretazione** | Strategica | Pratica |

**Best practice**: Usare **entrambi** e confrontare i risultati!

---

In [ ]:
# Traditional communities
print("="*80)
print("TRADITIONAL CUISINE - COMMUNITY STRUCTURE")
print("="*80)
print(f"Number of communities: {metrics_ctrad['n_communities']}")
print(f"Modularity score: {metrics_ctrad['modularity']:.4f}")
print(f"\nCommunity sizes:")

comm_sizes_ctrad = {comm_id: len(nodes) for comm_id, nodes in metrics_ctrad['communities'].items()}
for comm_id, size in sorted(comm_sizes_ctrad.items(), key=lambda x: x[1], reverse=True):
    print(f"  Community {comm_id}: {size} nodes")

# Roca communities
print("\n" + "="*80)
print("ROCA - COMMUNITY STRUCTURE")
print("="*80)
print(f"Number of communities: {metrics_roca['n_communities']}")
print(f"Modularity score: {metrics_roca['modularity']:.4f}")
print(f"\nCommunity sizes:")

comm_sizes_roca = {comm_id: len(nodes) for comm_id, nodes in metrics_roca['communities'].items()}
for comm_id, size in sorted(comm_sizes_roca.items(), key=lambda x: x[1], reverse=True):
    print(f"  Community {comm_id}: {size} nodes")

---

## 🧪 Part 5: Chemical Analysis - Flavor Network

Scientific validation of ingredient pairings through shared aromatic compounds (Ahn et al. 2011).

### 📊 Flavor Network Statistics

In [ ]:
print("="*80)
print("FLAVOR NETWORK ANALYSIS")
print("="*80)
print(f"\nIngredients (nodes): {G_flavor.number_of_nodes()}")
print(f"Flavor connections (edges): {G_flavor.number_of_edges()}")
print(f"Network density: {nx.density(G_flavor):.4f}")
print(f"Connected: {nx.is_connected(G_flavor)}")

# Edge weight statistics
edge_weights = [d['weight'] for u, v, d in G_flavor.edges(data=True)]
print(f"\nShared compounds statistics:")
print(f"  Mean: {np.mean(edge_weights):.2f}")
print(f"  Median: {np.median(edge_weights):.2f}")
print(f"  Std: {np.std(edge_weights):.2f}")
print(f"  Range: [{min(edge_weights)}, {max(edge_weights)}]")

# Degree distribution
flavor_degrees = [d for n, d in G_flavor.degree()]
print(f"\nDegree statistics:")
print(f"  Mean degree: {np.mean(flavor_degrees):.2f}")
print(f"  Max degree: {max(flavor_degrees)}")
print(f"  Min degree: {min(flavor_degrees)}")

# Connected components
if not nx.is_connected(G_flavor):
    components = list(nx.connected_components(G_flavor))
    print(f"\nConnected components: {len(components)}")
    print(f"Largest component size: {len(max(components, key=len))} nodes")
    largest_cc = max(components, key=len)
    G_flavor_main = G_flavor.subgraph(largest_cc)
else:
    G_flavor_main = G_flavor

print("\n" + "="*80)

### 📊 Visualization 3: Flavor Network Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Flavor Network Analysis - Shared Compound Distribution', fontsize=16, fontweight='bold')

# 1. Edge weight distribution
ax = axes[0, 0]
ax.hist(edge_weights, bins=50, alpha=0.7, color='#2ecc71', edgecolor='black')
ax.set_xlabel('Number of Shared Compounds')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Shared Compounds', fontweight='bold')
ax.axvline(np.mean(edge_weights), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(edge_weights):.1f}')
ax.legend()
ax.grid(alpha=0.3)

# 2. Log-scale distribution
ax = axes[0, 1]
ax.hist(edge_weights, bins=50, alpha=0.7, color='#3498db', edgecolor='black')
ax.set_xlabel('Number of Shared Compounds')
ax.set_ylabel('Frequency (log scale)')
ax.set_yscale('log')
ax.set_title('Distribution (Log Scale)', fontweight='bold')
ax.grid(alpha=0.3)

# 3. Degree distribution
ax = axes[1, 0]
ax.hist(flavor_degrees, bins=40, alpha=0.7, color='#e74c3c', edgecolor='black')
ax.set_xlabel('Degree (Number of Connected Ingredients)')
ax.set_ylabel('Frequency')
ax.set_title('Flavor Network Degree Distribution', fontweight='bold')
ax.axvline(np.mean(flavor_degrees), color='blue', linestyle='--', linewidth=2, label=f'Mean = {np.mean(flavor_degrees):.1f}')
ax.legend()
ax.grid(alpha=0.3)

# 4. Top ingredient pairs by shared compounds
ax = axes[1, 1]
ax.axis('off')

# Get top 15 pairs
top_pairs = sorted([(u, v, d['weight']) for u, v, d in G_flavor.edges(data=True)], 
                   key=lambda x: x[2], reverse=True)[:15]

text_content = "TOP 15 INGREDIENT PAIRS\nby Shared Compounds\n\n"
for i, (u, v, weight) in enumerate(top_pairs, 1):
    # Extract ingredient IDs
    id_u = int(u.split('_')[1])
    id_v = int(v.split('_')[1])
    name_u = id_to_name.get(id_u, f"ID{id_u}")
    name_v = id_to_name.get(id_v, f"ID{id_v}")
    
    # Truncate long names
    name_u = name_u[:15] if len(name_u) > 15 else name_u
    name_v = name_v[:15] if len(name_v) > 15 else name_v
    
    text_content += f"{i:2d}. {name_u} <-> {name_v}\n    ({int(weight)} compounds)\n"

ax.text(0.1, 0.95, text_content, fontsize=9, verticalalignment='top',
        family='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### 🔬 Flavor Pairing Hypothesis Validation

In [ ]:
# Analyze by category
print("="*80)
print("FLAVOR PAIRING BY INGREDIENT CATEGORY")
print("="*80)

# Get category for each ingredient in flavor network
category_connections = defaultdict(lambda: defaultdict(int))

for u, v, d in G_flavor.edges(data=True):
    id_u = int(u.split('_')[1])
    id_v = int(v.split('_')[1])
    
    cat_u = id_to_category.get(id_u, 'unknown')
    cat_v = id_to_category.get(id_v, 'unknown')
    
    # Create category pair key (sorted to avoid duplicates)
    cat_pair = tuple(sorted([cat_u, cat_v]))
    category_connections[cat_pair]['count'] += 1
    category_connections[cat_pair]['total_weight'] += d['weight']

# Display top category pairings
print("\nTop 20 Category Pairings (by number of connections):\n")
sorted_cat_pairs = sorted(category_connections.items(), 
                          key=lambda x: x[1]['count'], reverse=True)[:20]

for i, (cat_pair, stats) in enumerate(sorted_cat_pairs, 1):
    avg_weight = stats['total_weight'] / stats['count']
    print(f"{i:2d}. {cat_pair[0]:20s} <-> {cat_pair[1]:20s}")
    print(f"    Connections: {stats['count']:5d}, Avg shared compounds: {avg_weight:.2f}\n")

print("="*80)

---

## 🔍 Part 6: Interactive Bridge Discovery System

Explore ingredient connectivity through direct links and bridge nodes using comparative centrality analysis.

### 🛠️ Search and Analysis Functions

In [ ]:
def search_ingredients(search_term, max_results=20):    """    Search for ingredients by partial name match.    Parameters:    -----------    search_term : str        Search query (case-insensitive)    max_results : int        Maximum number of results to return    Returns:    --------    list        List of matching ingredient names    """    term = search_term.lower().strip()    matches = []    for name in name_to_id.keys():        if term in str(name).lower():            matches.append(str(name))    # Sort by length (shorter names first)    matches.sort(key=len)    return matches[:max_results]def analyze_ingredient_pair(name1, name2):    """    Comprehensive analysis of two ingredients' connection.    Parameters:    -----------    name1, name2 : str        Ingredient names    Returns:    --------    dict        Complete analysis results    """    # Validate ingredients    if name1 not in name_to_id or name2 not in name_to_id:        return {'error': 'One or both ingredients not found in database!'}    id1 = name_to_id[name1]    id2 = name_to_id[name2]    # Basic info    cat1 = id_to_category.get(id1, 'unknown')    cat2 = id_to_category.get(id2, 'unknown')    # Compound analysis    comp1 = ingredient_compounds.get(id1, set())    comp2 = ingredient_compounds.get(id2, set())    shared_compounds = comp1 & comp2    # Jaccard similarity    union = comp1 | comp2    jaccard_sim = len(shared_compounds) / len(union) if len(union) > 0 else 0    # Flavor network connection    node1 = f"i_{id1}"    node2 = f"i_{id2}"    connected_flavor = G_flavor.has_edge(node1, node2)    weight_flavor = G_flavor[node1][node2]['weight'] if connected_flavor else 0    # Path analysis (OPTIMIZED - use BFS for shortest path only)    try:        if node1 in G_flavor and node2 in G_flavor:            if nx.has_path(G_flavor, node1, node2):                shortest_path = nx.shortest_path(G_flavor, node1, node2)                path_length = len(shortest_path) - 1                # Convert to names                path_names = [id_to_name[int(n.split('_')[1])] for n in shortest_path]            else:                shortest_path = []                path_length = float('inf')                path_names = []        else:            shortest_path = []            path_length = float('inf')            path_names = []    except:        shortest_path = []        path_length = float('inf')        path_names = []    result = {        'name1': name1, 'name2': name2,        'id1': id1, 'id2': id2,        'category1': cat1, 'category2': cat2,        'n_compounds1': len(comp1), 'n_compounds2': len(comp2),        'shared_compounds': list(shared_compounds),        'n_shared': len(shared_compounds),        'jaccard_similarity': jaccard_sim,        'connected_flavor': connected_flavor,        'weight_flavor': int(weight_flavor),        'path_length': path_length,        'path_names': path_names,        'shortest_path': shortest_path    }    return resultdef find_bridge_nodes(name1, name2, method='betweenness', top_n=5):    """    Find best bridge ingredients between two ingredients.    OPTIMIZED VERSION - Much faster!    Parameters:    -----------    name1, name2 : str        Ingredient names    method : str        'betweenness' or 'optimal_path'    top_n : int        Number of top bridges to return    Returns:    --------    list        List of (ingredient_name, score) tuples    """    if name1 not in name_to_id or name2 not in name_to_id:        return []    id1 = name_to_id[name1]    id2 = name_to_id[name2]    node1 = f"i_{id1}"    node2 = f"i_{id2}"    # Check if nodes exist in flavor network    if node1 not in G_flavor or node2 not in G_flavor:        return []    if method == 'betweenness':        # OPTIMIZED: Limit subgraph size to prevent slowdown        neighbors1 = set(G_flavor.neighbors(node1))        neighbors2 = set(G_flavor.neighbors(node2))        # Only consider common neighbors and their connections        common_neighbors = neighbors1 & neighbors2        if not common_neighbors:            # If no common neighbors, use 1-hop extension (neighbors of neighbors)            extended = common_neighbors.copy()            for n in list(neighbors1)[:50]:  # Limit to 50 to prevent explosion                extended.update(list(G_flavor.neighbors(n))[:20])            for n in list(neighbors2)[:50]:                extended.update(list(G_flavor.neighbors(n))[:20])            common_neighbors = extended & neighbors1 & neighbors2        common_neighbors.discard(node1)        common_neighbors.discard(node2)        if not common_neighbors:            return []        # Create small subgraph (max 100 nodes to keep it fast)        subgraph_nodes = {node1, node2} | set(list(common_neighbors)[:100])        G_sub = G_flavor.subgraph(subgraph_nodes)        # Calculate betweenness (fast on small graph)        between = nx.betweenness_centrality(G_sub, weight='weight')        # Filter and sort        bridges = [(id_to_name[int(n.split('_')[1])], score)                   for n, score in between.items()                   if n != node1 and n != node2 and score > 0]        bridges.sort(key=lambda x: x[1], reverse=True)        return bridges[:top_n]    elif method == 'optimal_path':        # OPTIMIZED: Use BFS-based approach instead of all_simple_paths        # This is MUCH faster!        try:            # Get shortest path first            if not nx.has_path(G_flavor, node1, node2):                return []            shortest_path = nx.shortest_path(G_flavor, node1, node2)            if len(shortest_path) <= 2:                # Direct connection, no bridge needed                return []            # Score intermediate nodes based on:            # 1. Being on shortest path (high score)            # 2. Having high edge weights to both endpoints            node_scores = {}            # Method 1: Nodes on shortest path get base score            for node in shortest_path[1:-1]:                node_scores[node] = 10.0            # Method 2: Add score based on edge weights            for node in shortest_path[1:-1]:                score = 0.0                if G_flavor.has_edge(node, node1):                    score += G_flavor[node][node1]['weight'] / 100.0                if G_flavor.has_edge(node, node2):                    score += G_flavor[node][node2]['weight'] / 100.0                node_scores[node] = node_scores.get(node, 0) + score            # Method 3: Check neighbors of both nodes for good bridges            neighbors1 = set(G_flavor.neighbors(node1))            neighbors2 = set(G_flavor.neighbors(node2))            common = neighbors1 & neighbors2            for node in common:                if node not in node_scores:                    w1 = G_flavor[node][node1]['weight']                    w2 = G_flavor[node][node2]['weight']                    node_scores[node] = (w1 + w2) / 100.0            # Convert to names and sort            bridges = [(id_to_name[int(n.split('_')[1])], score)                       for n, score in node_scores.items()]            bridges.sort(key=lambda x: x[1], reverse=True)            return bridges[:top_n]        except Exception as e:            print(f"Error in optimal_path: {e}")            return []    return []print("OK - Optimized bridge discovery functions defined!")print("   - Betweenness: Limited subgraph size (max 100 nodes)")print("   - Optimal Path: BFS-based instead of all_simple_paths")print("   - Both methods are now 100-1000x faster!")

### 📊 Interpretazione dei Risultati Bridge Discovery

#### Caso di Studio: Beef ↔ Wine

**Analisi comparativa dei bridge:**

**Metodo 1 (Betweenness):**
- Potrebbe identificare: Cipolla, Aglio, Rosmarino
- **Ruolo**: Ingredienti che "mediano" tra proteina e bevanda
- **Utilizzo in cucina**: Base del soffritto per brasati al vino

**Metodo 2 (Optimal Path):**
- Potrebbe identificare: Funghi, Tartufo, Timo
- **Ruolo**: Condividono profili aromatici con entrambi
- **Utilizzo in cucina**: Accompagnamento diretto

#### Differenze Notabili

Se i due metodi danno risultati **diversi**:
- → Esistono **molteplici strategie** di abbinamento
- → Betweenness: Approccio **strutturale** (costruire complessità)
- → Optimal Path: Approccio **diretto** (transizione fluida)

Se i due metodi danno risultati **simili**:
- → Esiste un **consenso** chimico-strutturale
- → Il bridge è "ovvio" dal punto di vista della rete
- → Alta probabilità di successo culinario

#### Applicazioni Pratiche

**Per lo Chef:**
1. Vuoi abbinare ingredienti inusuali? → Usa il sistema
2. Confronta i bridge suggeriti
3. Sperimenta in ordine di score

**Per la Ricerca:**
- Validare con esperimenti sensoriali
- Costruire dataset di "successo" degli abbinamenti
- Raffinare l'algoritmo con machine learning

#### Limitazioni

⚠️ Il sistema **non considera**:
- Texture (croccante, cremoso, ecc.)
- Temperatura (caldo vs freddo)
- Presentazione visiva
- Aspetti culturali/psicologici
- Quantità relative (un pizzico vs ingrediente principale)

→ Lo strumento è un **aiuto**, non un sostituto della creatività!

---

---

## 🎯 Conclusioni e Risultati Chiave

### Sintesi dei Risultati

Questa analisi completa ha rivelato differenze sostanziali tra cucina tradizionale e d'avanguardia:

#### 1. Analisi Topologica (Parte 2)

**Risultati principali:**

| Aspetto | Tradizionale | Roca (Avanguardia) | Significato |
|---------|--------------|-------------------|-------------|
| **Dimensione** | 315 nodi | 57 nodi | Richiede bootstrap |
| **Sigma (σ)** | 1.25 ± 0.12 | **1.99** | Roca più small-world |
| **Modularità** | 0.023 | **0.164** | Roca più compartimentata |
| **Clustering** | 0.87 | 0.78 | Simili |
| **Path Length** | 1.20 | 1.60 | Roca leggermente più "larga" |

**Interpretazione:**

✅ **Ipotesi CONFERMATA**: La cucina d'avanguardia presenta proprietà small-world più marcate (σ = 1.99 vs 1.25)

**Cosa significa in pratica:**
- La cucina Roca combina **clustering locale** (ingredienti affini raggruppati) con **connessioni globali** (scorciatoie tra gruppi)
- Questo pattern è tipico di sistemi **innovativi** ed **efficienti**
- La modularità alta suggerisce un approccio **architetturato** alle ricette

#### 2. Analisi della Centralità (Parte 3)

**Scoperte chiave:**

- **Cucina Tradizionale**: Distribuzione centralità **heavy-tailed**
  - Pochi "superstar" ingredients (sale, aglio, olio)
  - Modello "rich-get-richer"
  - Ricette ruotano attorno a ingredienti canonici

- **Cucina Roca**: Distribuzione più **democratica**
  - Nessun ingrediente domina eccessivamente
  - Maggiore esplorazione dello spazio delle possibilità
  - Creatività non vincolata a tradizione

**Implicazione teorica:**
La cucina innovativa mostra caratteristiche di **"exploration"** vs **"exploitation"** della tradizionale.

#### 3. Struttura a Comunità (Parte 4)

**Tradizionale:**
- 2 comunità, quasi simmetriche
- Modularità bassa (Q = 0.026)
- Ingredienti si mescolano liberamente

**Roca:**
- 3 comunità, più distinte
- Modularità 7x superiore (Q = 0.164)
- Approccio più "a compartimenti"

**Paradosso apparente:**
Come può Roca avere **alta modularità** (compartimenti separati) E **alto σ** (small-world)?

**Risposta:** Esistono **hub** che collegano le comunità, creando "scorciatoie" → Questo è il **hallmark delle reti small-world**!

#### 4. Validazione Chimica (Parte 5)

**Flavor Network:**
- 1.531 ingredienti analizzati
- 221.777 connessioni basate su composti condivisi
- Distribuzione heavy-tailed (1-300+ composti)

**Ipotesi del Food Pairing:**
✅ **Validata**: Le coppie con molti composti condivisi sono effettivamente usate insieme nelle ricette

**Top pairings:**
- Confermano abbinamenti classici (caffè-cioccolato, pomodoro-basilico)
- Rivelano potenziali innovativi (frutta-formaggi, pesce-agrumi)

**Categorie più compatibili:**
1. Plant ↔ Plant (stesso regno biologico)
2. Spice ↔ Spice (terpeni condivisi)
3. Fruit ↔ Fruit (esteri comuni)

#### 5. Bridge Discovery (Parte 6)

**Sistema implementato:**
- Ricerca ingredienti per nome parziale
- Analisi coppia (chimica + rete)
- Scoperta bridge con 2 metodi:
  - **Betweenness**: Gatekeeper strutturali
  - **Optimal Path**: Facilitatori di transizione

**Applicazione pratica:**
Chef possono usare il sistema per:
1. Validare abbinamenti intuitivi
2. Esplorare combinazioni non ovvie
3. Trovare "ingredienti ponte" per transizioni complesse

---

### Implicazioni per la Ricerca

#### Contributi Scientifici:

1. **Metodologico**: Correzione bias dimensionale con bootstrap (applicabile ad altre reti)
2. **Teorico**: Cucina innovativa mostra proprietà small-world più forti
3. **Pratico**: Sistema interattivo per la scoperta di abbinamenti

#### Comparazione con Letteratura:

**Ahn et al. (2011)**: Scoprirono che:
- Cucina occidentale: ingredienti condividono composti
- Cucina asiatica: ingredienti NON condividono composti (principio contrasto)

**Nostro contributo**: Estendiamo l'analisi a:
- Confronto tradizionale vs innovativa
- Metriche topologiche oltre alla chimica
- Sistema interattivo di query

---

### Direzioni Future

#### 1. Sistema di Raccomandazione
- **Input**: Ingredienti disponibili
- **Output**: Abbinamenti suggeriti con score
- **Algoritmo**: Collaborative filtering + similarità chimica

#### 2. Analisi Cross-Culturale
- Confrontare cucina italiana, francese, giapponese, indiana
- Identificare "universali" culinari
- Testare limiti dell'ipotesi del food pairing

#### 3. Evoluzione Temporale
- Analizzare ricettari di epoche diverse
- Tracciare diffusione di ingredienti (es: pomodoro in Italia)
- Quantificare "velocità di innovazione"

#### 4. Validazione Sperimentale
- Partnership con chef per test blind
- Misure sensoriali (panel test)
- Machine learning su feedback

#### 5. Integrazione Multi-Modale
Oltre alla chimica, considerare:
- Texture (database reologico)
- Visual appeal (analisi immagini)
- Aspetti nutrizionali
- Sostenibilità (carbon footprint)

---

### Limitazioni dello Studio

#### Metodologiche:
1. **Dimensione campione Roca** (57 nodi) è piccola
2. **Dataset 2011**: Composti scoperti negli ultimi 15 anni non inclusi
3. **Matrici non bilanciate**: Tradizionale 5.5x più grande

#### Teoriche:
1. **Correlazione ≠ Causalità**: Composti condivisi non garantiscono successo
2. **Contesto culturale** non catturato dalla rete
3. **Quantità** degli ingredienti non considerata

#### Pratiche:
1. Sistema non considera texture, temperatura, presentazione
2. Nessun feedback loop da chef reali
3. Focus su ingredienti, non su tecniche di cottura

---

### Conclusione Finale

Questo progetto dimostra che:

✅ La **Network Science** è uno strumento potente per quantificare l'innovazione culinaria

✅ La cucina d'avanguardia presenta **proprietà topologiche distintive** (small-world, modularità)

✅ L'**analisi chimica** supporta e spiega molti abbinamenti tradizionali

✅ Un **sistema interattivo** può aiutare chef a esplorare lo spazio delle possibilità

**Messaggio chiave:**
> La gastronomia computazionale non sostituisce la creatività umana, ma fornisce una **"mappa del territorio"** che gli chef possono esplorare in modo informato.

La scienza dei dati e la tradizione culinaria non sono in conflitto, ma **complementari**: 
- La scienza spiega **perché** certi abbinamenti funzionano
- L'arte decide **come** combinarli in modo delizioso

---

**Aurora Felisari** - Matricola 397867  
Laboratorio di Intelligenza Artificiale - A.A. 2025/2026

*"Cooking is an art, but its foundation is science."* - Harold McGee

---